# NeuroX v4 - Fast GPU Training (Google Colab T4)

## Train ALL 17 models in 5-15 minutes instead of 12+ hours!

**How it works:**
- Colab gives you a free NVIDIA T4 GPU (8.1 TFLOPS vs ~0.1 on CPU)
- GPU batch size = 256 (vs 32 on CPU) = 8x fewer gradient steps per epoch
- Total speedup: ~80-100x faster than your Windows trading PC

**Crash-proof training:**
- Each model checkpoint is saved to Google Drive immediately after training
- If Colab crashes, just re-run - already-trained models are skipped automatically
- No more losing hours of progress!

**Instructions:**
1. Make sure GPU is enabled: Runtime -> Change runtime type -> T4 GPU
2. Click "Runtime -> Run all" (or run cells one by one)
3. Authorize Google Drive access when prompted (Step 1)
4. Wait ~10 minutes for all 17 models to train
5. Checkpoints are already on your Drive - download from there to your trading PC

**No CSV upload needed** - data is downloaded from Yahoo Finance automatically!

**IMPORTANT:** This notebook clones the `fast-gpu-training` branch which contains
the NeuroX v4 code and the `train_gpu_fast.py` script. The `main` branch does NOT
have these files.


In [ ]:
#@title Step 0: Verify GPU is available (MUST show T4)
import subprocess
import sys

print("=" * 60)
print("  STEP 0: GPU Verification")
print("=" * 60)

try:
    import torch
except ImportError:
    print("PyTorch not found, will be installed in Step 2.")
    print("For now, checking nvidia-smi...")
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    if result.returncode == 0:
        print(result.stdout)
        print("\nGPU detected via nvidia-smi. Proceeding.")
    else:
        raise RuntimeError(
            "No GPU detected! Go to: Runtime -> Change runtime type -> T4 GPU\n"
            "Then click 'Runtime -> Restart and run all'"
        )
else:
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"  GPU: {gpu_name}")
        print(f"  VRAM: {vram:.1f} GB")
        print(f"  CUDA: {torch.version.cuda}")
        print()
        print("  GPU is ready! Training will be FAST.")
    else:
        raise RuntimeError(
            "No GPU detected! Go to: Runtime -> Change runtime type -> T4 GPU\n"
            "Then click 'Runtime -> Restart and run all'"
        )

print("=" * 60)


In [ ]:
#@title Step 1: Mount Google Drive (protects checkpoints from crashes)
import os

print("=" * 60)
print("  STEP 1: Mount Google Drive")
print("=" * 60)
print()
print("  Mounting Drive so checkpoints are saved after EACH model.")
print("  If Colab crashes, your progress is safe on Drive!")
print()

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print()
    print("  Drive mounted successfully!")
    print("  Checkpoints will auto-save to:")
    print("    /content/drive/MyDrive/neurox_checkpoints/")
except ImportError:
    print("  Not running in Colab - Drive mount skipped.")
    print("  Checkpoints will be saved locally only.")
except Exception as e:
    print(f"  Drive mount failed: {e}")
    print("  Training will continue without Drive backup.")
    print("  (Checkpoints saved locally - lost if Colab crashes)")

print()
print("=" * 60)


In [ ]:
#@title Step 2: Clone repo (fast-gpu-training branch) & Install dependencies (~2 min)
import os
import subprocess

print("=" * 60)
print("  STEP 2: Clone & Install")
print("=" * 60)

REPO_URL = "https://github.com/gagandocx/Claude.git"
BRANCH = "fast-gpu-training"
CLONE_DIR = "/content/Claude"
WORK_DIR = "/content/Claude/NeuroX/neurox_v4"

# ---- Clone or update the repo ----
if not os.path.exists(CLONE_DIR):
    print(f"\n[1/3] Cloning branch '{BRANCH}' from {REPO_URL}...")
    result = subprocess.run(
        ["git", "clone", "-b", BRANCH, "--single-branch", REPO_URL, CLONE_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"STDERR: {result.stderr}")
        raise RuntimeError(f"git clone failed! Error: {result.stderr}")
    print("  Repo cloned successfully!")
else:
    print(f"\n[1/3] Repo already exists. Pulling latest from '{BRANCH}'...")
    # Make sure we're on the right branch
    subprocess.run(["git", "checkout", BRANCH], cwd=CLONE_DIR, capture_output=True)
    subprocess.run(["git", "pull", "origin", BRANCH], cwd=CLONE_DIR, capture_output=True)
    print("  Repo updated!")

# ---- Verify critical files exist ----
print("\n[2/3] Verifying project files...")
critical_files = [
    os.path.join(WORK_DIR, "train_gpu_fast.py"),
    os.path.join(WORK_DIR, "config", "settings.py"),
    os.path.join(WORK_DIR, "data", "market_data.py"),
    os.path.join(WORK_DIR, "models"),
]
missing = []
for f in critical_files:
    exists = os.path.exists(f)
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {f}")
    if not exists:
        missing.append(f)

if missing:
    print("\n" + "!" * 60)
    print("  CRITICAL ERROR: Required files are missing!")
    print("  This means the branch clone failed or the branch is wrong.")
    print("\n  Attempting to fix by re-cloning...")
    import shutil
    shutil.rmtree(CLONE_DIR, ignore_errors=True)
    result = subprocess.run(
        ["git", "clone", "-b", BRANCH, "--single-branch", REPO_URL, CLONE_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"Failed to clone branch '{BRANCH}'. Error: {result.stderr}\n"
            f"Make sure the branch exists at: {REPO_URL}"
        )
    # Re-verify
    still_missing = [f for f in critical_files if not os.path.exists(f)]
    if still_missing:
        raise RuntimeError(
            f"Files still missing after re-clone: {still_missing}\n"
            f"The branch '{BRANCH}' may not contain the NeuroX code."
        )
    print("  Fixed! All files present after re-clone.")

# ---- Install Python dependencies ----
print("\n[3/3] Installing Python dependencies...")
print("  (torch with CUDA, scikit-learn, lightgbm, catboost, xgboost, etc.)")

deps = [
    "torch",
    "scikit-learn==1.9.0",
    "lightgbm",
    "catboost",
    "xgboost",
    "yfinance",
    "ta",
    "pandas",
    "numpy",
    "tqdm",
    "joblib",
    "hmmlearn",
    "scipy",
]

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + deps,
    capture_output=True, text=True
)
if result.returncode != 0:
    print(f"  pip install output: {result.stdout}")
    print(f"  pip install errors: {result.stderr}")
    # Don't fail hard - some packages might already be installed
    print("  WARNING: Some packages may have failed. Will try to continue.")
else:
    print("  All dependencies installed successfully!")

# ---- Final verification ----
print("\n" + "=" * 60)
print("  Setup complete!")
print(f"  Working directory: {WORK_DIR}")
print(f"  Branch: {BRANCH}")
print("  Ready to train.")
print("=" * 60)


In [ ]:
#@title Step 3: Train ALL 17 models on GPU (~5-15 min) - saves checkpoints incrementally!
import os
import subprocess
import sys

print("=" * 60)
print("  STEP 3: Training ALL 17 Models (crash-proof!)")
print("=" * 60)

WORK_DIR = "/content/Claude/NeuroX/neurox_v4"

# Verify we're in the right place
train_script = os.path.join(WORK_DIR, "train_gpu_fast.py")
if not os.path.exists(train_script):
    raise RuntimeError(
        f"train_gpu_fast.py not found at {train_script}!\n"
        "Did Step 2 complete successfully? Re-run Step 2 first."
    )

print(f"\n  Script: {train_script}")
print(f"  Working dir: {WORK_DIR}")
print(f"  This will take ~5-15 minutes on a T4 GPU.")
print()
print("  CRASH-PROOF: Each model is saved immediately after training.")
print("  If Colab disconnects, just re-run this cell to resume!")
print("  Already-trained models will be loaded from checkpoints.")

# Check for existing checkpoints (resume info)
ckpt_dir = os.path.join(WORK_DIR, "checkpoints")
drive_dir = "/content/drive/MyDrive/neurox_checkpoints"
if os.path.isdir(drive_dir) and os.listdir(drive_dir):
    print(f"\n  Found existing checkpoints on Drive ({len(os.listdir(drive_dir))} files)")
    print("  Will resume training from where it left off!")
elif os.path.isdir(ckpt_dir) and os.listdir(ckpt_dir):
    print(f"\n  Found existing local checkpoints ({len(os.listdir(ckpt_dir))} files)")
    print("  Will resume training from where it left off!")

print(f"\n  Watch the progress bars below...\n")

# Run training with real-time output
os.chdir(WORK_DIR)

# Use os.system for real-time output in Colab (subprocess captures it)
exit_code = os.system(f"cd {WORK_DIR} && {sys.executable} train_gpu_fast.py")

if exit_code != 0:
    print("\n" + "!" * 60)
    print("  Training failed!")
    print("  Common fixes:")
    print("  1. Make sure GPU is enabled (Runtime -> Change runtime type -> T4)")
    print("  2. Restart runtime (Runtime -> Restart runtime) and re-run all cells")
    print("  3. Check if Colab session ran out of RAM (try again in a new session)")
    print()
    print("  NOTE: Any models that finished before the crash are already saved!")
    print("  Just re-run this cell to resume from where it stopped.")
    print("!" * 60)
    raise RuntimeError("Training script exited with errors. See output above.")
else:
    print("\n" + "=" * 60)
    print("  Training COMPLETE!")
    print("  All checkpoints saved incrementally during training.")
    if os.path.isdir("/content/drive/MyDrive"):
        print("  Checkpoints are on Google Drive: /content/drive/MyDrive/neurox_checkpoints/")
    else:
        print("  Checkpoints saved locally: /content/Claude/NeuroX/neurox_v4/checkpoints/")
    print("=" * 60)


In [ ]:
#@title Step 4: Download trained checkpoints to your PC
import os
import shutil

print("=" * 60)
print("  STEP 4: Package & Download Checkpoints")
print("=" * 60)

checkpoint_dir = "/content/Claude/NeuroX/neurox_v4/checkpoints"
drive_dir = "/content/drive/MyDrive/neurox_checkpoints"

# Use Drive checkpoints if available (they survive crashes)
if os.path.isdir(drive_dir) and os.listdir(drive_dir):
    src_dir = drive_dir
    print(f"\n  NOTE: Checkpoints are already on your Google Drive!")
    print(f"  Location: Google Drive > neurox_checkpoints/")
    print(f"  You can download them directly from drive.google.com")
    print()
    print("  Or use the zip download below...")
elif os.path.isdir(checkpoint_dir) and os.listdir(checkpoint_dir):
    src_dir = checkpoint_dir
else:
    raise RuntimeError(
        "No checkpoints found! Training may have failed.\n"
        "Re-run Step 3."
    )

# List all saved checkpoint files
files_list = sorted(os.listdir(src_dir))
if not files_list:
    raise RuntimeError(
        "Checkpoints directory is empty! Training did not save any models.\n"
        "Re-run Step 3 and check for errors."
    )

print("\nSaved checkpoints:")
total_size = 0
for f in files_list:
    fpath = os.path.join(src_dir, f)
    size = os.path.getsize(fpath)
    total_size += size
    print(f"  {f:<40} {size / 1024:.0f} KB")

print(f"\n  Total: {len(files_list)} files, {total_size / (1024*1024):.1f} MB")

# Create zip archive
zip_path = "/content/neurox_checkpoints"
print(f"\nCreating zip archive...")
shutil.make_archive(zip_path, "zip", src_dir)
zip_size = os.path.getsize(f"{zip_path}.zip") / (1024 * 1024)
print(f"  Archive: {zip_path}.zip ({zip_size:.1f} MB)")

# Download
print("\nStarting download...")
try:
    from google.colab import files
    files.download(f"{zip_path}.zip")
    print("\n  Download started! Check your browser downloads.")
except ImportError:
    print("  Not running in Colab - skipping auto-download.")
    print(f"  Zip file saved at: {zip_path}.zip")
except Exception as e:
    print(f"  Auto-download failed: {e}")
    print("  Your checkpoints are already on Google Drive - download from there.")

print("\n" + "=" * 60)
print("  DONE! After downloading, unzip into your trading PC:")
print("  neurox_v4/checkpoints/")
print("\n  TIP: If Drive is mounted, you can also access checkpoints at:")
print("  Google Drive > neurox_checkpoints/")
print("=" * 60)


## Troubleshooting

**"No GPU detected"**
- Go to Runtime -> Change runtime type -> Select T4 GPU -> Save
- Then Runtime -> Restart and run all

**"train_gpu_fast.py not found"**
- The clone may have used the wrong branch. Delete and re-clone:
  ```python
  !rm -rf /content/Claude
  ```
- Then re-run from Step 2

**"ModuleNotFoundError"**
- Re-run Step 2 to install dependencies
- If `config.settings` or `data.market_data` is missing, the branch is wrong

**Training runs out of memory / Colab crashes**
- Just re-run from Step 3! Models that already finished are saved.
- The script will automatically skip completed models and resume.
- If Drive was mounted, checkpoints survive even runtime restarts.

**Download doesn't work**
- Checkpoints are already on Google Drive (if you mounted it in Step 1)
- Go to drive.google.com > neurox_checkpoints/ and download from there

## How to retrain (weekly)

Run this notebook once a week to keep your models fresh with latest market data:
1. Open this notebook in Colab
2. Runtime -> Run all
3. Checkpoints auto-save to Drive during training
4. Download neurox_checkpoints/ from Drive to your trading PC
5. Replace checkpoints/ folder on your trading PC
6. Restart your EA

The models will train on the latest 7 days of 1-minute data, 60 days of 15-minute
data, and 2 years of hourly data from Yahoo Finance, ensuring they stay adapted to
current market conditions.

**TIP:** To force retraining all models from scratch (ignore existing checkpoints),
delete the checkpoint folder first:
```python
!rm -rf /content/Claude/NeuroX/neurox_v4/checkpoints
!rm -rf /content/drive/MyDrive/neurox_checkpoints
```
